In [148]:
load('IrreducibilityLeanProofWriter.sage')
load('RingOfIntegersLeanProofWriter.sage')
%run CertificateClassGroupLean2.0.ipynb
%run PrimalityCertificate.ipynb
%run InvariantProofs.ipynb

In [149]:
R.<X>= PolynomialRing(ZZ)

#Input defining polynomial. 
T = X^5 - 20*X^3 - 50*X^2 - 45*X - 26 
K.<a> = NumberField(T)

#Input basis for the ring of integers. 
B = [1, a, a^2, 1/2*a^3 - 1/2*a, 1/4*a^4 - 1/4*a^3 - 1/4*a^2 - 1/4*a - 1/2]
B = [K(b) for b in B]
num = '30c'

In [150]:
#Create initial files: 

#Proof of irreducibility of defining polynomial. 
doc = open(f"Irreducible{num}.lean", "w")

#Definition of the number field, definition of subalgebra and basis, and p-maximality proofs. 
f = open(f"RI{num}.lean", "w")

#Write RI (Ring of integers) file. Output is a list of 'bad' primes that do not satisfy Dedekind Criterion, 
# flagl (1 if p unramified), flagW (1 if a wtiness was found), flagD (1 if discriminant ought to be computed) 
bad, flagl, flaglW, flagD = LeanProof(T,B,f,f'Irreducible{num}',num, '',1)

#Write Irreducible file, with proof of irreducibility
LeanProofIrreducible(T, doc)

In [151]:
import contextlib

#List of primes involved in computation of ring of integers. 
F = [factor(T.discriminant())[i][0] for i in range(len(factor(T.discriminant())))] 

#We compute the Minkowski bound: 
M = K.minkowski_bound()

#The number of rational primes below M 
PP = [p for p in primes(floor(M) + 1)]
m = len(PP)

#The number of primes ideals above a single prime is (on average, and assuming Galois group S_n) the harmonic number Hn.  
#Hence, in each lean file with prime ideals we put around 20 ideals so that we consider batches of 20/Hn rational primes. 
NI = ceil(20 / harmonic_number(K.degree()))

#Number of files in which we divide the generation of prime ideals: 
NF = ceil(m/NI)

#Number of intervals we use for the collection certificate. 
number_interval = ceil(NF/2)

t = open(f"PrimesBelowCert{num}.lean", "w")

with contextlib.redirect_stdout(t):
    print(f"""import IdealArithmetic.Examples.NF{num}.PrimesBelow{num}F{NF-1}
noncomputable section""")
    listI, listP = PrimesBelowBoundCertificteGenInterval(K,B,ceil(M),number_interval)
    t.close()

for i in range(NF): 
    f = open(f"PrimesBelow{num}F{i}.lean", "w")
    with contextlib.redirect_stdout(f):
        if i == 0: 
            print(f"""
import IdealArithmetic.IdealArithmetic
import IdealArithmetic.Examples.NF{num}.RI{num}
import IdealArithmetic.ClassGroupGeneration

open Classical Polynomial

noncomputable section """)
            PrimesBelowBound(K, B, PP[i*NI], PP[min((i+1) * NI - 1,len(PP) -1)] , F)
        else : 
            print(f"""
import IdealArithmetic.Examples.NF{num}.PrimesBelow{num}F{i-1}

open Classical Polynomial

noncomputable section """)
            PrimesBelowBound(K, B, PP[i*NI], PP[min((i+1) * NI - 1,len(PP) -1)] , F)
            print("")
    f.close()



In [152]:
#We compute a generator for the class group and the corresponding data to certify its non-principality. 

import re
O = K.ring_of_integers()
Cl = K.class_group('pari')
n = K.degree()

#The generators for an ideal generating the class group. 
ideal_gens = Cl.gens()[0].ideal().gens_reduced()

#The ideal generating the class group. 
J = O.ideal(ideal_gens)

#The order of this (cyclic) class group. 
t = Cl(J).order()
x = (J ^ t).gens_reduced()[0]
pr = t.prime_divisors()[0]

A , Q , elems , flag = MatrixPrimes (K, O, pr, x, 200) #flag = 1 for the case that pr divides the torsion order. 
elems = [K(r) for r in elems]
#The primes ideals of degree 1 to certify the non-principality. 
Ql = [list(Q[i].gens()) for i in range(len(Q))]

#The rational prime below these prime ideals. 
Qlq = [Ql[i][0] for i in range(len(Ql))]

if flag == 0 : 
    names = ['zeta' + str(i + 1) for i in range(len(Q) - 1)] + ['alpha']
else : 
    names = ['zeta' + str(i + 1) for i in range(len(Q) - 2)] + ['v'] + ['alpha']

In [153]:
#Write the file with the certificate of nonprincipality of the generator J

set_random_seed(10)
l = open(f"NonPrincipalGenerator{num}.lean", "w")
with contextlib.redirect_stdout(l):
    L, gi = NonPrincipalityCertLean(K, B, num, Ql, pr, elems, names, ideal_gens, 'J', x, t)
l.close()
print(gi)

[[1], (2, a), (4, a + 2), (-a - 2,)]


In [154]:
def name_cert_example_cyclic(l) : 
    return (gi[0])[l[0]]

In [155]:
# We generate the file with the relations between the prime ideals of norm below M and the generator of the 
# class group J. 

#set_random_seed(10)
r = open(f"RelationIdeals{num}.lean", "w")
with contextlib.redirect_stdout(r):
    print(f"""import IdealArithmetic.Examples.NF{num}.PrimesBelow{num}F{NF - 1}
import IdealArithmetic.Examples.NF{num}.NonPrincipalGenerator{num}

noncomputable section""")
    BM = RelationsGenerator(K, B, ceil(M), [ideal_gens], 'J', name_cert_example, name_cert_example_cyclic)
r.close()

In [156]:
%%capture capInv

print(f"""import IdealArithmetic.Examples.NF{num}.NonPrincipalGenerator{num}
import Mathlib.NumberTheory.NumberField.ClassNumber
import IdealArithmetic.ResultantRecursive
import IdealArithmetic.DiscriminantSubalgebraBuilder
import IdealArithmetic.ClassGroupGeneration
import IdealArithmetic.Examples.NF{num}.RelationIdeals{num}
import IdealArithmetic.Examples.NF{num}.PrimesBelowCert{num}

/- Number field `K(α)` with with α root of polynomial `{T}`.
Class number `{t}`, generated by class of ideal `J = ({ideal_gens[0]}, {ideal_gens[1]})`. We have `J ^ {pr}` is principal. -/

/- Ring of integers with basis `{B}` -/

open BigOperators Classical Matrix Polynomial

noncomputable section

instance hirr : Fact $ (Irreducible (map (algebraMap ℤ ℚ) T)) where
  out :=  (Polynomial.Monic.irreducible_iff_irreducible_map_fraction_map (T_monic)).1 T_irreducible """)

print(f"""
noncomputable instance K_field : Field K := by
  unfold K
  infer_instance

instance K_numberField : NumberField K := by
  unfold K
  infer_instance

lemma K_finrank : Module.finrank ℚ K = {len(B)} := by
  unfold K
  rw [Module.finrank_eq_card_basis (AdjoinRoot.powerBasisAux _), Polynomial.natDegree_map_eq_of_injective, T_degree]
  · simp
  · exact RingHom.injective_int (algebraMap ℤ ℚ)
  · exact Irreducible.ne_zero hirr.out """)

print(f"""
theorem O_integral_closure : O = integralClosure ℤ K := by
  refine eq_of_piMaximal_at_all_primes_int O Om hm ?_
  intro p hp
  by_cases hc : p ∈ {bad}
  · fin_cases hc""")
for p in bad:
    if flagl[p] == 0:
        if flaglW[p] == 1:
            print(f'    exact pMaximal_of_MaximalOrderCertificateWLists {p} O Om hm M{p}')
        else : 
            print(f'    exact pMaximal_of_MaximalOrderCertificateLists {p} O Om hm M{p}')
    else : 
        print(f'    exact pMaximal_of_MaximalOrderCertificateOfUnramifiedLists {p} O Om hm M{p}')
print(f"""  · haveI : Fact $ Nat.Prime p := fact_iff.2 hp
    refine piMaximal_of_root_in_order_of_satisfiesDedekindCriterion_int Adj T_monic hm ?_ hroot_mem
     (satisfiesDedekindAlmostAllLists_of_certificate T _ T_ofList {bad} D p hp hc)
    rw [T_degree, rank_subalgebra_eq_card_basis Om B']
""")
print("")
print(f"""theorem  O_ringOfIntegers' : O = NumberField.RingOfIntegers K := by rw [O_integral_closure] ; rfl
""")

print("""

instance : Module.Finite ℤ (Additive ((↥O)ˣ ⧸ CommGroup.torsion (↥O)ˣ)) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFiniteIntAdditiveQuotientUnitsRingOfIntegersSubgroupTorsion K

instance : Module.Free ℤ (Additive ((↥O)ˣ ⧸ CommGroup.torsion (↥O)ˣ)) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFreeIntAdditiveQuotientUnitsRingOfIntegersSubgroupTorsion K

instance :  Fintype ↥(CommGroup.torsion (↥O)ˣ) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFintypeSubtypeUnitsRingOfIntegersMemSubgroupTorsion K

instance : IsCyclic ↥(CommGroup.torsion (↥O)ˣ) := by
  rw [O_integral_closure]
  exact NumberField.Units.instIsCyclicSubtypeUnitsRingOfIntegersMemSubgroupTorsion K

instance DD' : IsDedekindDomain O  := by
  rw [O_integral_closure]
  exact integralClosure.isDedekindDomain ℤ ℚ K

instance : Module.Free ℤ ↥O := Module.Free.of_basis B 

""")
RankUnitCertificateLean(R(T), 'T_ofList', 'Adj', 'O_integral_closure', 'RC')
print('')
if flag == 0:
    pNeDvdTorsionLean(K, B, 'K_finrank' , 'O_integral_closure', 'IC', pr)
    print('')
    CertificateNonPrincipalityNT (L, pr, t, names[:-1], 'I', 'J', 'alpha', Qlq, 'RC', 'J' + str(t))
    print("""
theorem J_not_principal : ¬ ∃ b, J = Ideal.span {b} :=
  not_principal_of_NonPrincipalCertificateNDvdT NPCJ """)
else : 
    print('')
    CertificateNonPrincipalityDVDT (L, p, t, names[:-2], names[len(names) - 2], order_v, 'I', 'J', 'alpha', Qlq, 'RC', 'J' + str(t))
    print("""
theorem J_not_principal : ¬ ∃ b, J = Ideal.span {b} :=
  not_principal_of_NonPrincipalCertificateDvdT NPCJ """)

if flagD == 1:
    D = K.discriminant()
    print(f"""
lemma T_discr : T.discriminant = {T.discriminant()} :=  by
  convert discriminant_eq_DiscriminantOfPRemainder_of_SturmBuilderOfList SturmRC
  rw [T_ofList]

theorem K_discr : NumberField.discr K = {D} := by
  rw [discr_numberField_eq_discrSubalgebraBuilder T_irreducible BQ O_integral_closure]
  rw [T_discr]
  rfl

lemma K_nrComplexPlaces : NumberField.InfinitePlace.nrComplexPlaces K = {K.signature()[1]} := by
  rw [nrComplexPlaces_of_RankUnitsCertificate RC]
  rfl

theorem K_minowski : MinkowskiBound K ≤ ({float(K.minkowski_bound() + 0.01)} : ℝ) := by
  refine K_minkowski_decimal _ ?_
  rw [K_nrComplexPlaces, K_discr, K_finrank]
  norm_num

""")

ClassGroupPrimeOrderProof(listI,listP, BM,f'PB{ceil(M)}',ceil(M), 'J', 'J_not_principal',pr,'J' + str(pr))

In [157]:
with open(f"Invariants{num}.lean", "w") as f:
    f.write(capInv.stdout)
f.close()